# Convolutional Neural Networks from Scratch

**Module 04: CNN**  
**Objective**: Understand and implement CNNs from first principles

CNNs revolutionized computer vision by:
- Learning spatial hierarchies of features
- Translation invariance through weight sharing
- Reducing parameters via local connectivity

## What You'll Learn
1. Convolution operation (forward and backward)
2. Pooling layers
3. Building CNN architectures
4. Understanding receptive fields
5. Feature visualization
6. Common CNN architectures (LeNet, AlexNet concepts)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Convolution Operation

**2D Convolution**: Slide a kernel/filter over an image and compute dot products.

$$(\mathbf{I} * \mathbf{K})[i, j] = \sum_m \sum_n \mathbf{I}[i+m, j+n] \cdot \mathbf{K}[m, n]$$

**Why convolution?**
- **Weight sharing**: Same kernel applied everywhere
- **Local connectivity**: Each neuron sees only a local patch
- **Translation invariance**: Detect features regardless of position

In [ ]:
def convolve2d(image, kernel, padding=0, stride=1):
    """
    2D convolution (valid mode).
    
    Args:
        image: (H, W) or (C, H, W)
        kernel: (H_k, W_k) or (C, H_k, W_k)
        padding: pad size
        stride: stride size
    
    Returns:
        output: convolved image
    """
    if len(image.shape) == 2:
        image = image[np.newaxis, :, :]
    if len(kernel.shape) == 2:
        kernel = kernel[np.newaxis, :, :]
    
    C, H, W = image.shape
    C_k, H_k, W_k = kernel.shape
    
    # Add padding
    if padding > 0:
        image = np.pad(image, ((0, 0), (padding, padding), (padding, padding)), mode='constant')
        _, H, W = image.shape
    
    # Output dimensions
    H_out = (H - H_k) // stride + 1
    W_out = (W - W_k) // stride + 1
    
    output = np.zeros((H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride
            w_start = j * stride
            
            # Extract patch
            patch = image[:, h_start:h_start+H_k, w_start:w_start+W_k]
            
            # Convolve
            output[i, j] = np.sum(patch * kernel)
    
    return output

# Test convolution
image = np.random.randn(28, 28)
kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]])  # Vertical edge detector

output = convolve2d(image, kernel, padding=1, stride=1)

print(f"Input shape: {image.shape}")
print(f"Kernel shape: {kernel.shape}")
print(f"Output shape: {output.shape}\n")

# Visualize edge detection
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Create simple image with edges
simple_image = np.zeros((50, 50))
simple_image[:, 15:35] = 1.0

# Edge detectors
vertical_edge = np.array([[1, 0, -1],
                          [2, 0, -2],
                          [1, 0, -1]])

horizontal_edge = np.array([[1, 2, 1],
                            [0, 0, 0],
                            [-1, -2, -1]])

# Apply convolutions
vertical_output = convolve2d(simple_image, vertical_edge, padding=1)
horizontal_output = convolve2d(simple_image, horizontal_edge, padding=1)

axes[0].imshow(simple_image, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(vertical_output, cmap='gray')
axes[1].set_title('Vertical Edge Detection')
axes[1].axis('off')

axes[2].imshow(horizontal_output, cmap='gray')
axes[2].set_title('Horizontal Edge Detection')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 2. Pooling Layers

**Max Pooling**: Take maximum in each region
**Average Pooling**: Take average

**Purpose**:
- Reduce spatial dimensions
- Provide translation invariance
- Reduce overfitting

In [ ]:
def max_pool2d(image, pool_size=2, stride=2):
    """Max pooling operation."""
    if len(image.shape) == 2:
        image = image[np.newaxis, :, :]
    
    C, H, W = image.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    
    output = np.zeros((C, H_out, W_out))
    
    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                h_start = i * stride
                w_start = j * stride
                
                patch = image[c, h_start:h_start+pool_size, w_start:w_start+pool_size]
                output[c, i, j] = np.max(patch)
    
    if C == 1:
        return output[0]
    return output

def avg_pool2d(image, pool_size=2, stride=2):
    """Average pooling operation."""
    if len(image.shape) == 2:
        image = image[np.newaxis, :, :]
    
    C, H, W = image.shape
    H_out = (H - pool_size) // stride + 1
    W_out = (W - pool_size) // stride + 1
    
    output = np.zeros((C, H_out, W_out))
    
    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                h_start = i * stride
                w_start = j * stride
                
                patch = image[c, h_start:h_start+pool_size, w_start:w_start+pool_size]
                output[c, i, j] = np.mean(patch)
    
    if C == 1:
        return output[0]
    return output

# Test pooling
test_image = np.random.randn(8, 8)
max_pooled = max_pool2d(test_image, pool_size=2, stride=2)
avg_pooled = avg_pool2d(test_image, pool_size=2, stride=2)

print(f"Original shape: {test_image.shape}")
print(f"Max pooled shape: {max_pooled.shape}")
print(f"Avg pooled shape: {avg_pooled.shape}\n")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(test_image, cmap='viridis')
axes[0].set_title('Original (8×8)')
axes[0].axis('off')

axes[1].imshow(max_pooled, cmap='viridis')
axes[1].set_title('Max Pooling (4×4)')
axes[1].axis('off')

axes[2].imshow(avg_pooled, cmap='viridis')
axes[2].set_title('Average Pooling (4×4)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 3. Backpropagation Through Convolution

**Forward**: $Y = X * W$

**Backward**:
- $\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} * W^T$ (flip kernel)
- $\frac{\partial L}{\partial W} = X * \frac{\partial L}{\partial Y}$

In [ ]:
class Conv2D:
    """Convolutional layer with backpropagation."""
    
    def __init__(self, in_channels, out_channels, kernel_size, padding=0, stride=1):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.padding = padding
        self.stride = stride
        
        # Initialize weights (He initialization)
        self.weights = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.bias = np.zeros(out_channels)
        
        self.cache = {}
    
    def forward(self, X):
        """Forward pass."""
        batch_size, C, H, W = X.shape
        
        # Pad if needed
        if self.padding > 0:
            X_padded = np.pad(X, ((0, 0), (0, 0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')
        else:
            X_padded = X
        
        # Output dimensions
        H_out = (H + 2*self.padding - self.kernel_size) // self.stride + 1
        W_out = (W + 2*self.padding - self.kernel_size) // self.stride + 1
        
        output = np.zeros((batch_size, self.out_channels, H_out, W_out))
        
        # Convolve
        for b in range(batch_size):
            for f in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * self.stride
                        w_start = j * self.stride
                        
                        patch = X_padded[b, :, h_start:h_start+self.kernel_size, w_start:w_start+self.kernel_size]
                        output[b, f, i, j] = np.sum(patch * self.weights[f]) + self.bias[f]
        
        self.cache['X'] = X
        self.cache['X_padded'] = X_padded
        return output
    
    def backward(self, dY):
        """Backward pass."""
        X = self.cache['X']
        X_padded = self.cache['X_padded']
        
        batch_size, C, H, W = X.shape
        _, _, H_out, W_out = dY.shape
        
        # Initialize gradients
        dX_padded = np.zeros_like(X_padded)
        dW = np.zeros_like(self.weights)
        db = np.zeros_like(self.bias)
        
        # Compute gradients
        for b in range(batch_size):
            for f in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * self.stride
                        w_start = j * self.stride
                        
                        # Gradient w.r.t. weights
                        patch = X_padded[b, :, h_start:h_start+self.kernel_size, w_start:w_start+self.kernel_size]
                        dW[f] += dY[b, f, i, j] * patch
                        
                        # Gradient w.r.t. input
                        dX_padded[b, :, h_start:h_start+self.kernel_size, w_start:w_start+self.kernel_size] += dY[b, f, i, j] * self.weights[f]
                
                # Gradient w.r.t. bias
                db[f] += np.sum(dY[:, f, :, :])
        
        # Remove padding from dX
        if self.padding > 0:
            dX = dX_padded[:, :, self.padding:-self.padding, self.padding:-self.padding]
        else:
            dX = dX_padded
        
        return dX, dW, db

# Test gradients
print("Testing Conv2D gradients...\n")

conv = Conv2D(in_channels=1, out_channels=2, kernel_size=3, padding=1, stride=1)
X_test = np.random.randn(2, 1, 8, 8)  # batch_size=2, 1 channel, 8x8

# Forward
Y = conv.forward(X_test)
print(f"Input shape: {X_test.shape}")
print(f"Output shape: {Y.shape}")

# Backward
dY = np.random.randn(*Y.shape)
dX, dW, db = conv.backward(dY)
print(f"dX shape: {dX.shape}")
print(f"dW shape: {dW.shape}")
print(f"db shape: {db.shape}")

## 4. Simple CNN for Digit Classification

In [ ]:
class SimpleCNN:
    """Simple CNN: Conv → ReLU → Pool → FC → Softmax"""
    
    def __init__(self):
        # Conv: 1×8×8 → 4×6×6 (kernel=3, no padding)
        self.conv = Conv2D(in_channels=1, out_channels=4, kernel_size=3, padding=0, stride=1)
        
        # After pooling: 4×3×3 = 36
        # FC: 36 → 10
        self.fc_weights = np.random.randn(36, 10) * 0.01
        self.fc_bias = np.zeros(10)
        
        self.losses = []
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def forward(self, X):
        """Forward pass."""
        # Conv + ReLU
        conv_out = self.conv.forward(X)
        relu_out = self.relu(conv_out)
        
        # Max pooling (2×2)
        batch_size, channels, H, W = relu_out.shape
        pooled = np.zeros((batch_size, channels, H//2, W//2))
        
        for b in range(batch_size):
            for c in range(channels):
                pooled[b, c] = max_pool2d(relu_out[b, c], pool_size=2, stride=2)
        
        # Flatten
        flattened = pooled.reshape(batch_size, -1)
        
        # FC + Softmax
        fc_out = flattened @ self.fc_weights + self.fc_bias
        probs = self.softmax(fc_out)
        
        self.cache = {
            'X': X,
            'conv_out': conv_out,
            'relu_out': relu_out,
            'pooled': pooled,
            'flattened': flattened,
            'fc_out': fc_out,
            'probs': probs
        }
        
        return probs
    
    def compute_loss(self, y_true, y_pred):
        """Cross-entropy loss."""
        m = y_true.shape[0]
        log_probs = -np.log(y_pred[range(m), y_true] + 1e-8)
        return np.sum(log_probs) / m
    
    def backward(self, y_true, learning_rate=0.01):
        """Simplified backward (only updates FC for speed)."""
        m = y_true.shape[0]
        
        # Output gradient
        dfc_out = self.cache['probs'].copy()
        dfc_out[range(m), y_true] -= 1
        dfc_out /= m
        
        # FC gradients
        dfc_weights = self.cache['flattened'].T @ dfc_out
        dfc_bias = np.sum(dfc_out, axis=0)
        
        # Update FC only (full CNN backprop is slow in pure NumPy)
        self.fc_weights -= learning_rate * dfc_weights
        self.fc_bias -= learning_rate * dfc_bias
    
    def train(self, X, y, epochs=10, learning_rate=0.1):
        """Training loop."""
        for epoch in range(epochs):
            # Forward
            probs = self.forward(X)
            loss = self.compute_loss(y, probs)
            self.losses.append(loss)
            
            # Backward
            self.backward(y, learning_rate)
            
            # Accuracy
            preds = np.argmax(probs, axis=1)
            acc = np.mean(preds == y)
            
            print(f"Epoch {epoch+1}/{epochs}: Loss={loss:.4f}, Acc={acc:.4f}")
    
    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)

# Load digits dataset (8×8 images)
digits = load_digits()
X = digits.data.reshape(-1, 1, 8, 8) / 16.0  # Normalize
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n" + "="*50)
print("Training Simple CNN on Digits Dataset\n")
print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

# Train
model = SimpleCNN()
model.train(X_train, y_train, epochs=20, learning_rate=0.5)

# Test
y_pred= model.predict(X_test)
test_acc = np.mean(y_pred == y_test)
print(f"\nTest Accuracy: {test_acc:.4f}")

# Plot
plt.figure(figsize=(10, 4))
plt.plot(model.losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

# Visualize predictions
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    ax = axes[i//5, i%5]
    ax.imshow(X_test[i, 0], cmap='gray')
    ax.set_title(f'True: {y_test[i]}, Pred: {y_pred[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Receptive Field

**Receptive field**: Region of input that affects a particular output neuron.

**Formula**: $r_{out} = r_{in} + (k - 1) \times \prod_{i} s_i$

Where:
- $r_{in}$: input receptive field
- $k$: kernel size
- $s_i$: stride at layer $i$

In [ ]:
def compute_receptive_field(layers):
    """
    Compute receptive field for CNN architecture.
    
    Args:
        layers: list of (kernel_size, stride, padding) tuples
    
    Returns:
        receptive_field: size of receptive field
    """
    rf = 1  # Start with 1×1
    
    for i, (kernel_size, stride, padding) in enumerate(layers):
        # Compute jump (stride product up to this layer)
        jump = np.prod([s for _, s, _ in layers[:i]])
        
        # Update receptive field
        rf += (kernel_size - 1) * jump
    
    return rf

# Example: Simple CNN
layers = [
    (3, 1, 0),  # Conv 3×3, stride 1
    (2, 2, 0),  # Pool 2×2, stride 2
    (3, 1, 0),  # Conv 3×3, stride 1
    (2, 2, 0),  # Pool 2×2, stride 2
]

rf = compute_receptive_field(layers)
print(f"Receptive field: {rf}×{rf}")
print("\nLayer-by-layer receptive fields:")

cumulative_rf = 1
for i, (k, s, p) in enumerate(layers):
    jump = np.prod([stride for _, stride, _ in layers[:i]])
    cumulative_rf += (k - 1) * jump
    print(f"After layer {i+1} (k={k}, s={s}): {cumulative_rf}×{cumulative_rf}")

## Summary

You've learned CNNs from scratch:
- ✅ Convolution operation (forward & backward)
- ✅ Pooling layers (max & average)
- ✅ Building CNN architectures
- ✅ Backpropagation through conv layers
- ✅ Receptive field computation

**Key insights**:
1. Convolutions learn spatial hierarchies
2. Pooling provides translation invariance
3. Weight sharing dramatically reduces parameters
4. Deeper networks have larger receptive fields

**Next Notebook**: Modern CNNs with PyTorch (ResNet, BatchNorm, Transfer Learning)